# Native CLM v0 — M3L-1 Historical Address-State Capacity

Checkpoint-only capacity/family diagnostic. This notebook does **not** retrain Native CLM, update Cells/router/certificates, grow topology, or consume new formal continual-learning seeds. It reconstructs the exact M3R snapshot/checkpoints used by M3L and measures the same lineage-local gate across diagonal, rank-8/16/32/64/128, full-covariance Gaussian address states, plus the same offline linear oracle.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m3l1-address-state-capacity'
REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
DATA = Path('/kaggle/working/native-clm-m3r-address-data')
CHECKPOINTS = Path('/kaggle/working/native-clm-m3r-address-checkpoints')
OUT = ROOT / 'artifacts/experiments/native-clm-v0-m3l1-address-state-capacity'

def run(cmd, cwd=None):
    print('+', ' '.join(str(x) for x in cmd), flush=True)
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

if not ROOT.exists():
    run(['git', 'clone', REPO_URL, str(ROOT)])
run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'], cwd=ROOT)
run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], cwd=ROOT)


In [ ]:
from kaggle_secrets import UserSecretsClient
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'HF_TOKEN is missing'
assert os.environ['GITHUB_TOKEN'], 'GITHUB_TOKEN is missing'
assert torch.cuda.is_available(), 'CUDA is required for the canonical diagnostic'
assert torch.cuda.device_count() >= 2, f'expected >=2 GPUs, got {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Boundary: checkpoint-only; no Native CLM training; M3R seeds 73611/73612/73613 are consumed inputs only.')


In [ ]:
# Reconstruct and byte/SHA-verify the exact M3R A/B/C/D snapshot used by M3L.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m3r_address_data.py', '--output-dir', DATA], cwd=ROOT)
manifest = json.loads((DATA / 'manifest.json').read_text())
assert manifest['exact_parent_snapshot_verified'] is True
assert manifest['parent_manifest_sha256'] == '213ddb9d093ea44fd0524e6ba6318f86a61c54270bd5cad6ddeb3233470565b0'
print('Exact M3R data snapshot: PASS')


In [ ]:
# Download the exact three published M3R lineage checkpoints and verify SHA-256 identities.
run([sys.executable, 'scripts/research/fetch_native_clm_v0_m3r_address_checkpoints.py', '--output-dir', CHECKPOINTS], cwd=ROOT)
ckpt_manifest = json.loads((CHECKPOINTS / 'manifest.json').read_text())
assert ckpt_manifest['revision'] == 'a23b521e137a7e44616809895d44d87cc7d6f87f'
assert sorted(r['seed'] for r in ckpt_manifest['records']) == [73611, 73612, 73613]
print('Published M3R lineage checkpoints: PASS')


In [ ]:
# Run the frozen M3L-1 capacity curve. GPU0/GPU1 process two consumed checkpoints concurrently; the first free GPU receives the third.
run([
    sys.executable,
    'scripts/research/run_native_clm_v0_m3l1_address_state_capacity.py',
    '--data-dir', DATA,
    '--checkpoint-dir', CHECKPOINTS,
    '--output-dir', OUT,
    '--devices', 'cuda:0,cuda:1',
], cwd=ROOT)
result = json.loads((OUT / 'diagnostic-result.json').read_text())
curve = result['capacity_curve']
print(json.dumps({
    'classification': result['classification'],
    'valid_edges': f"{result['valid_edge_count']}/{result['edge_count']}",
    'oracle_median_auc': result['offline_oracle']['median'],
    'rank0': curve['rank-0']['median'],
    'rank8': curve['rank-8']['median'],
    'rank16': curve['rank-16']['median'],
    'rank32': curve['rank-32']['median'],
    'rank64': curve['rank-64']['median'],
    'rank128': curve['rank-128']['median'],
    'full_covariance': curve['full-covariance']['median'],
    'minimum_passing_low_rank': result['minimum_passing_low_rank'],
    'full_covariance_passes': result['full_covariance_passes'],
    'rank16_identity': result['rank16_parent_identity']['passed'],
}, indent=2))


In [ ]:
# Publish JSON/CSV/MD only. M3L-1 creates no model checkpoint.
run([
    sys.executable,
    'scripts/research/publish_native_clm_v0_m3l1_address_state_capacity.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
], cwd=ROOT)
print('Published M3L-1 classification:', result['classification'])


## Interpretation boundary

`LOW_RANK_CAPACITY_SUFFICIENT` means a finite registered rank recovers the original M3L feasibility gates and licenses that address-state capacity for a future newly registered online continual-language experiment. `FULL_COVARIANCE_REQUIRED` means the second-order family survives but low rank through 128 is insufficient. `GAUSSIAN_FAMILY_LIMITED` means increasing covariance rank alone is not enough. None of these classifications rewrites M3L/M3R/M3 outcomes.